# どこまで見えてるの？ — ニューラルネットワークの入力感度分析

ニューラルネットワークは「どの入力をどれだけ見ているか」を**ヤコビアン**で定量化できます。

## ヤコビアンとは

ネットワーク $f: \mathbb{R}^{n} \to \mathbb{R}^{m}$ のヤコビアン行列 $J$ は次のように定義されます。

$$
J = \frac{\partial f(x)}{\partial x}
\in \mathbb{R}^{m \times n},
\quad
J_{ij} = \frac{\partial y_i}{\partial x_j}
$$

各エントリ $J_{ij}$ は「出力 $y_i$ が入力 $x_j$ をどれだけ見ているか」を表します。

## 入力感度

入力 $x_j$ の出力全体への感度は、ヤコビアンの $j$ 列の L2 ノルムで定義します。

$$
s_j = \left\| \frac{\partial f}{\partial x_j} \right\|_2
= \sqrt{\sum_{i=1}^{m} J_{ij}^2}
$$

$s_j$ が大きいほど、$x_j$ はネットワークの出力に強く影響します。

---

このノートブックでは以下を可視化します。

1. **ヤコビアン行列のヒートマップ** — どの入力がどの出力に影響するか
2. **入力感度の棒グラフ** — 各入力の「見え方」の強さ
3. **標準 NN と ICNN の比較** — ネットワーク構造による違い

In [ ]:
# 必要なライブラリのインポート
from __future__ import annotations

import sys
import pathlib

# python/ ディレクトリを PYTHONPATH に追加
repo_root = pathlib.Path('.').resolve().parent
python_path = repo_root / 'python'
if str(python_path) not in sys.path:
    sys.path.insert(0, str(python_path))

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

from jax_util.base import DEFAULT_DTYPE
from jax_util.neuralnetwork import build_neuralnetwork
from jax_util.neuralnetwork.jacobian import compute_jacobian, input_sensitivity

# 乱数シードを固定して再現性を確保
KEY = jax.random.PRNGKey(0)

print(f'JAX version: {jax.__version__}')
print(f'Default dtype: {DEFAULT_DTYPE}')

## 1. 標準 NN のヤコビアン可視化

3 入力 → 2 出力の標準全結合 NN を構築し、
各入力が各出力にどれだけ影響を与えるかをヒートマップで確認します。

In [ ]:
# 標準 NN の構築
INPUT_DIM = 3
HIDDEN_DIM = 8
OUTPUT_DIM = 2
BATCH_SIZE = 5

model_std = build_neuralnetwork(
    network_type='standard',
    layer_sizes=(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM),
    activation='tanh',
    random_key=KEY,
)

# 入力データを生成（形状: input_dim × batch_size）
x = jax.random.normal(KEY, (INPUT_DIM, BATCH_SIZE), dtype=DEFAULT_DTYPE)

# ヤコビアン計算: (batch_size, output_dim, input_dim)
J_std = compute_jacobian(model_std, x)
print(f'ヤコビアン形状: {J_std.shape}  (batch_size, output_dim, input_dim)')

In [ ]:
# 最初のバッチ要素のヤコビアン行列をヒートマップで可視化
fig, axes = plt.subplots(1, BATCH_SIZE, figsize=(3 * BATCH_SIZE, 3), sharey=True)

vmax = float(jnp.abs(J_std).max())

for b, ax in enumerate(axes):
    im = ax.imshow(
        np.array(J_std[b]),  # (output_dim, input_dim)
        cmap='RdBu_r',
        vmin=-vmax,
        vmax=vmax,
        aspect='auto',
    )
    ax.set_title(f'batch {b}', fontsize=10)
    ax.set_xlabel('入力 $x_j$')
    ax.set_xticks(range(INPUT_DIM))
    ax.set_xticklabels([f'$x_{j}$' for j in range(INPUT_DIM)])

axes[0].set_ylabel('出力 $y_i$')
axes[0].set_yticks(range(OUTPUT_DIM))
axes[0].set_yticklabels([f'$y_{i}$' for i in range(OUTPUT_DIM)])

fig.colorbar(im, ax=axes[-1], label='$\\partial y_i / \\partial x_j$')
fig.suptitle('標準 NN のヤコビアン行列（各バッチ要素）', fontsize=13)
plt.tight_layout()
plt.show()

print('赤 = 正の影響（増加）、青 = 負の影響（減少）')

## 2. 入力感度の可視化

感度 $s_j = \| \partial f / \partial x_j \|_2$ は、
入力 $x_j$ が出力**全体**に与える影響の「強さ」です。
値が大きいほど、そのネットワークは $x_j$ をより「強く見ている」と言えます。

In [ ]:
# 入力感度: (batch_size, input_dim)
s_std = input_sensitivity(model_std, x)
print(f'感度行列形状: {s_std.shape}  (batch_size, input_dim)')

# バッチ全体の平均感度
s_mean = np.array(jnp.mean(s_std, axis=0))  # (input_dim,)
s_std_err = np.array(jnp.std(s_std, axis=0))

fig, ax = plt.subplots(figsize=(5, 4))
x_pos = np.arange(INPUT_DIM)

bars = ax.bar(x_pos, s_mean, yerr=s_std_err, capsize=5, color='steelblue', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'$x_{j}$' for j in range(INPUT_DIM)], fontsize=12)
ax.set_ylabel('感度 $s_j = \\|\\partial f / \\partial x_j\\|_2$', fontsize=11)
ax.set_title('標準 NN の入力感度（バッチ平均 ± 標準偏差）', fontsize=12)
ax.set_ylim(0, None)

# 値を棒グラフ上に表示
for bar, val in zip(bars, s_mean):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 3. 標準 NN と ICNN の比較

ICNN（Input Convex Neural Network）は入力に対して凸な関数を表現します。
その特性上、ヤコビアン（＝劣微分）は単調性を持ちます。
標準 NN との感度パターンの違いを見てみましょう。

In [ ]:
# ICNN の構築（最終出力次元は 1 が一般的）
model_icnn = build_neuralnetwork(
    network_type='icnn',
    layer_sizes=(INPUT_DIM, HIDDEN_DIM, 1),
    activation='softplus',
    random_key=KEY,
)

J_icnn = compute_jacobian(model_icnn, x)  # (batch_size, 1, input_dim)
s_icnn = input_sensitivity(model_icnn, x)  # (batch_size, input_dim)

print(f'ICNN ヤコビアン形状: {J_icnn.shape}')

# 2 モデルの感度を並べて比較
s_std_mean = np.array(jnp.mean(input_sensitivity(model_std, x), axis=0))
s_icnn_mean = np.array(jnp.mean(s_icnn, axis=0))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, s_mean, title, color in zip(
    axes,
    [s_std_mean, s_icnn_mean],
    ['標準 NN (tanh)', 'ICNN (softplus)'],
    ['steelblue', 'tomato'],
):
    ax.bar(x_pos, s_mean, color=color, alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'$x_{j}$' for j in range(INPUT_DIM)], fontsize=12)
    ax.set_ylabel('感度 $s_j$', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(0, max(s_std_mean.max(), s_icnn_mean.max()) * 1.2)
    for j, val in enumerate(s_mean):
        ax.text(j, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('入力感度の比較：標準 NN vs ICNN', fontsize=13)
plt.tight_layout()
plt.show()

## 4. 勾配の変化：入力空間全体での感度マップ

2 次元入力 ($x_0, x_1$) を持つネットワークの場合、
入力空間全体にわたって感度がどう変化するかを等高線図で確認できます。

$$
s(x) = \left\| \nabla_x f(x) \right\|_2
$$

In [ ]:
# 2D 入力 → 1 出力のネットワークで感度マップを描画
model_2d = build_neuralnetwork(
    network_type='standard',
    layer_sizes=(2, 16, 1),
    activation='tanh',
    random_key=KEY,
)

# 2D グリッドを生成
n_grid = 40
grid_range = jnp.linspace(-3.0, 3.0, n_grid)
xx, yy = jnp.meshgrid(grid_range, grid_range)  # (n_grid, n_grid)

# グリッド点を (2, n_grid^2) の形式にまとめてバッチ処理
x_grid = jnp.stack([xx.ravel(), yy.ravel()], axis=0)  # (2, n_grid^2)

# 全グリッド点の出力値と感度を計算
y_grid = model_2d(x_grid)  # (1, n_grid^2)
s_grid = input_sensitivity(model_2d, x_grid)  # (n_grid^2, 2)
s_norm = np.array(jnp.linalg.norm(s_grid, axis=-1).reshape(n_grid, n_grid))
y_vals = np.array(y_grid[0].reshape(n_grid, n_grid))
xx_np, yy_np = np.array(xx), np.array(yy)

# 感度マップ + 等値線を描画
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左：感度（どこを強く見ているか）
cf0 = axes[0].contourf(xx_np, yy_np, s_norm, levels=30, cmap='hot_r')
fig.colorbar(cf0, ax=axes[0], label='感度 $\\|\\nabla_x f\\|_2$')
axes[0].set_title('入力感度マップ $\\|\\nabla_x f(x)\\|_2$', fontsize=12)
axes[0].set_xlabel('$x_0$')
axes[0].set_ylabel('$x_1$')

# 右：出力値（ネットワークが何を出力するか）
cf1 = axes[1].contourf(xx_np, yy_np, y_vals, levels=30, cmap='RdBu_r')
fig.colorbar(cf1, ax=axes[1], label='$f(x)$')
axes[1].set_title('ネットワーク出力 $f(x)$', fontsize=12)
axes[1].set_xlabel('$x_0$')
axes[1].set_ylabel('$x_1$')

fig.suptitle('2D 入力空間での感度マップと出力値', fontsize=13)
plt.tight_layout()
plt.show()

print('感度が高い領域 = ネットワークが入力変化に最も敏感な場所（活性化関数の非線形性が強い領域）')

## まとめ

| 指標 | 意味 | 利用場面 |
|------|------|----------|
| $J_{ij} = \partial y_i / \partial x_j$ | 出力 $i$ の入力 $j$ への感度（符号付き） | どの入力がどの出力に寄与するか調べる |
| $s_j = \|\partial f / \partial x_j\|_2$ | 入力 $j$ の全出力への影響の大きさ | 重要な入力特徴量を特定する |
| $\|\nabla_x f(x)\|_2$ | 入力点 $x$ での局所的感度 | 入力空間の「見え方マップ」を描く |

全結合 NN では**構造的には全入力が全出力に接続**されていますが、
**感度の大小**によって「実質的にどこを見ているか」が分かります。

この分析は `jax_util.neuralnetwork.jacobian` モジュールで提供しています。

```python
from jax_util.neuralnetwork.jacobian import compute_jacobian, input_sensitivity

J = compute_jacobian(network, x)    # (batch, output_dim, input_dim)
s = input_sensitivity(network, x)   # (batch, input_dim)
```